In [1]:
from gnncloudmanufacturing.data import read_fatahi_dataset
from gnncloudmanufacturing.random_solver import random_solve
from gnncloudmanufacturing.validation import total_cost_from_graph, check_feasibility, total_cost_from_gamma
from gnncloudmanufacturing.utils import delta_from_gamma, graph_from_problem, gamma_from_target, delta_from_gamma
from gnncloudmanufacturing.graph_model import ConvGNN, os_type, ss_type

import numpy as np
from tqdm.auto import trange, tqdm
from time import time
import pandas as pd
import torch

In [2]:
def predict(model, dataset, n_operations):
    problem_name = []
    total_cost = []
    comp_time = []
    for problem in tqdm(dataset):
        start = time()
        total = np.inf
        for i in range(10):
            graph = graph_from_problem(problem, max_operations=n_operations)
            graph.edata['feat'][os_type][:, 0] /= 10
            graph.edata['feat'][ss_type][:] /= 100
            pred = model.predict(graph)
            gamma = gamma_from_target(pred, graph, problem)
            delta = delta_from_gamma(problem, gamma)
            check_feasibility(gamma, delta, problem)
            _total = total_cost_from_gamma(problem, gamma, delta).item()
            if total > _total:
                total = _total
        total_cost.append(total)
        problem_name.append(problem['name'])
        comp_time.append(time() - start)
    return pd.DataFrame({'problem_name': problem_name, 'total_cost': total_cost, 'comp_time': comp_time}).round(2)

In [3]:
n_tasks, n_operations, n_cities = 5, 20, 20
dataset = read_fatahi_dataset(
    '../../data/fatahi.xlsx', 
    sheet_names=[
        f'{n_tasks},{n_operations},{n_cities}-1',
        f'{n_tasks},{n_operations},{n_cities}-2', 
        f'{n_tasks},{n_operations},{n_cities}-3',
    ]
)
for problem in dataset:
    print(f'Problem: {problem["name"]}')

  0%|          | 0/3 [00:00<?, ?it/s]

Problem: 5,20,20-1
Problem: 5,20,20-2
Problem: 5,20,20-3


In [4]:
model = ConvGNN.load_from_checkpoint(
    checkpoint_path=f'gnn-{n_tasks}-{n_operations}-{n_cities}.ckpt',
    ins_dim=1,
    ino_dim=n_operations,
    out_dim=32,
    n_layers=3,
    lr=0.002,
)
model.eval()

ConvGNN(
  (enc): ModuleList(
    (0): ConvLayer(
      (W_os): Linear(in_features=22, out_features=32, bias=True)
      (W_ss): Linear(in_features=2, out_features=32, bias=True)
      (W_in): Linear(in_features=20, out_features=32, bias=True)
      (W_self): Linear(in_features=20, out_features=32, bias=True)
      (W_out): Linear(in_features=20, out_features=32, bias=True)
      (W_o): Linear(in_features=96, out_features=32, bias=True)
    )
    (1-2): 2 x ConvLayer(
      (W_os): Linear(in_features=34, out_features=32, bias=True)
      (W_ss): Linear(in_features=33, out_features=32, bias=True)
      (W_in): Linear(in_features=32, out_features=32, bias=True)
      (W_self): Linear(in_features=32, out_features=32, bias=True)
      (W_out): Linear(in_features=32, out_features=32, bias=True)
      (W_o): Linear(in_features=96, out_features=32, bias=True)
    )
  )
  (dropout): Dropout(p=0.0, inplace=False)
  (dec): DotProductDecoder()
)

In [5]:
predict(model, dataset, n_operations)

  0%|          | 0/3 [00:00<?, ?it/s]

,problem_name,total_cost,comp_time
0,"5,20,20-1",16818.86,0.86
1,"5,20,20-2",17553.00,0.82
2,"5,20,20-3",19331.85,0.83
